# Module 7: Handling Errors

**Utrains Python Fundamentals** &middot; lab notebook

Part 4 of the course: Writing Reliable Code.

## What you will be able to do by the end

- Catch a failure with try and except instead of crashing
- Use else and finally for success-only and always-run code
- Retry a flaky call instead of giving up on the first failure
- Raise your own errors when your code detects a problem

## How to use this notebook

Run every cell in order with **Shift + Enter**. Read the markdown before each
block, then run the code and compare what you see against what you expected.

Two cells in this notebook are marked **Your turn**. They contain `____` where
a piece of the syntax is missing. They will fail if you run them as they are.
That is deliberate. Replace each `____`, then run the cell until it succeeds.

The last section is the **Lab**. It is a short task with no code written for
you, so you have to put the module together yourself.

When something goes wrong, for example a value cannot be converted or a network
call times out, Python stops the program and raises an error. `try` and
`except` catch that error so the program can respond instead of crashing.

## try and except

In [ ]:
import json

api_response = '{"id": "msg_01", "content": "Success"}'

try:
    data = json.loads(api_response)
    print("parsed fine:", data["content"])
except json.JSONDecodeError:
    print("Failed to parse API response.")

In [ ]:
broken_response = '{"id": "msg_01", "content": '   # truncated on the wire

try:
    data = json.loads(broken_response)
    print("parsed fine:", data["content"])
except json.JSONDecodeError:
    print("Failed to parse API response.")

You can catch more than one kind of error, and use `else` for code that should
only run on success and `finally` for code that must always run.

In [ ]:
try:
    result = 10 / 0
except ZeroDivisionError:
    print("Cannot divide by zero.")
except ValueError:
    print("That was not a valid value.")
else:
    print("Success:", result)      # only if no exception happened
finally:
    print("Done trying.")          # always runs, error or not

> `finally` is for cleanup work, such as closing a connection, that has to
> happen whether or not something went wrong.

---

### Your turn 1

An incident ticket must always end up marked closed, even when the update step blows up. Pick the right two keywords.

Replace each `____` below, then run the cell. It will not run until you do.

In [ ]:
def update_ticket(ticket_id):
    raise ConnectionError("ticketing system unreachable")

ticket = {"id": "INC-4412", "status": "open"}

try:
    update_ticket(ticket["id"])
    print("ticket updated")
except ConnectionError as e:
    print("could not update:", e)
finally:
    ticket["status"] = "closed"

print(ticket)

## Retrying instead of giving up

Calling an AI model is a network call, and network calls fail sometimes: the
connection drops, the server is slow, or you hit a rate limit. Wrapping the
call lets your program recover instead of stopping.

`time.sleep()` pauses between attempts. Module 10 covers `import` in full.

In [ ]:
import time


def call_model(prompt):
    if prompt == "":
        raise TimeoutError("model did not respond in time")
    return f"response to: {prompt}"


def call_with_retry(prompt, attempts=3):
    for attempt in range(1, attempts + 1):
        try:
            return call_model(prompt)
        except TimeoutError:
            print(f"attempt {attempt} timed out, retrying...")
            time.sleep(0.2)
    raise RuntimeError("model call failed after all retries")


print(call_with_retry("Summarize this document."))

try:
    call_with_retry("")
except RuntimeError as e:
    print("gave up:", e)

## Raising your own errors

You are not limited to errors Python raises on its own. `raise` lets you signal
that something in your own code has gone wrong, with a message explaining why.

In [ ]:
def set_age(age):
    if age < 0:
        raise ValueError("age cannot be negative")
    return age


try:
    set_age(-5)
except ValueError as e:
    print("Invalid input:", e)

---

### Your turn 2

Write a guard that rejects an invalid model temperature. Anything outside 0.0 to 2.0 should raise, with a clear message.

Replace each `____` below, then run the cell. It will not run until you do.

In [ ]:
def set_temperature(value):
    if value < 0.0 or value > 2.0:
        raise ValueError(f"temperature {value} is outside 0.0 to 2.0")
    return value


print("valid:", set_temperature(0.7))

try:
    set_temperature(3.5)
except ValueError as e:
    print("rejected:", e)

---

## Lab: A deployment step that refuses to crash


Write a function `deploy(stage)` that raises a `RuntimeError` when the stage
name is `"migrate"`, and otherwise returns a success message.

Then write a loop over the stages `build`, `test`, `migrate`, `release` that
calls `deploy()` for each one. A failing stage must not stop the others.

For every stage, record the outcome in a results dictionary. Use `finally` so
that a line is always printed for each stage, whether it succeeded or not. At
the end, print how many stages succeeded and how many failed.

For extra credit, wrap the `migrate` stage in the retry pattern from earlier in
this notebook and give it three attempts before recording it as failed.


**Done when:**

- [ ] deploy() raises for one specific stage
- [ ] One failing stage does not stop the loop
- [ ] finally guarantees a line per stage
- [ ] A summary counts successes and failures

Write your answer in the cell below. There is no starter code on purpose.

In [ ]:
stages = ["build", "test", "migrate", "release"]

# Your lab answer goes here.

---

## Practice exercises

Work through these on your own after the lab. They come straight from the
course reference guide, so the wording matches what you will see there.

1. Wrap a deployment step function in try/except to catch a simulated DeploymentError and print a clear message.
2. Write a retry loop around a function that checks whether a cloud storage bucket exists, catching a simulated exception.
3. Use try/except/finally so an incident ticket always gets marked as closed, even if updating it raises an error.
4. Add retry logic around a function that simulates calling a model API which sometimes raises a TimeoutError.

---

*Utrains &middot; support@utrains.org &middot; https://utrains.org*